# Canadian Rental Market Pressure Analysis

This notebook runs the SQL analysis phase of the Canadian Rental Market Pressure project. It processes Table 1.0 CMHC Rental Market Survey data from the local SQLite database to compute market tightness indices, data-driven thresholds, and city segmentations.

### 1. Target Audience
- **Housing-policy analysts** monitoring affordability and supply tightness trends.
- **Municipal housing planners** designing local housing strategies and programs.
- **Renter advocacy organizations** seeking empirical data to support tenant policy revisions.

### 2. Decision Supported
- Identifies which major Canadian rental markets should be prioritized for deeper monitoring because they combine high rents, low vacancy, or worsening conditions.

### 3. Metric Selection Rationale
- **Average Rent (Two-Bedroom):** A standardized measure of general rent levels.
- **Vacancy Rate:** Direct measure of immediate rental supply tightness.
- **Turnover Rate:** Indicator of tenant mobility and market churn.
- **Rent Growth (YoY):** Trend indicator showing how fast prices are rising.
- **Vacancy Change (YoY):** Indicator showing if supply conditions are easing or tightening.

### 4. Important Context & Assumptions
- **Screening, Not Affordability:** This is a market-tightness screening tool. It is not an affordability score because it does not incorporate income data.
- **Standardized Two-Bedroom Benchmark:** Average two-bedroom rent is used as the default standardized comparison measure for Version 1. It is a standardized comparison benchmark across markets and does not represent all renters or all unit sizes.
- **No Causal Claims:** This analysis highlights correlations and market states. It does not establish causal housing relationships.
- **No Advice:** This tool is for regional monitoring, not for recommending where individual renters should live.
- **Caution-Quality Data Handling:** Estimates with caution-quality indicator 'd' are kept in all tables but are excluded from the calculation of data-driven thresholds to prevent high-uncertainty values from distorting thresholds.

In [1]:
import os
import sqlite3
import pandas as pd

# Ensure output directories exist
os.makedirs('../outputs/tables', exist_ok=True)

# Database connection
db_path = '../database/rental_market.db'
conn = sqlite3.connect(db_path)

## Query 1: 2025 Market Overview

**Business Question:** Which major Canadian rental markets have the highest rents in 2025, and what are their corresponding vacancy and turnover rates?

In [2]:
q1_sql = """
SELECT 
    centre,
    vacancy_rate_2025,
    turnover_rate_2025,
    average_rent_2br_2025,
    rent_growth_pct_2024_2025,
    vacancy_change_pct_points,
    is_caution_quality
FROM rental_market_2024_2025
WHERE has_valid_rent_2025 = 1 
  AND has_valid_vacancy_2025 = 1
ORDER BY average_rent_2br_2025 DESC;
"""

df_rankings = pd.read_sql_query(q1_sql, conn)
df_rankings.to_csv('../outputs/tables/rental_market_rankings.csv', index=False)
print(f"Exported {len(df_rankings)} rows to outputs/tables/rental_market_rankings.csv")
df_rankings.head(15)

Exported 43 rows to outputs/tables/rental_market_rankings.csv


,centre,vacancy_rate_2025,turnover_rate_2025,average_rent_2br_2025,rent_growth_pct_2024_2025,vacancy_change_pct_points,is_caution_quality
0,Vancouver CMA,3.7,11.6,2363.0,2.2,2.1,0
1,Victoria CMA,3.3,18.8,2120.0,5.1,0.7,0
2,Kelowna CMA,6.4,29.6,2118.0,2.4,2.6,0
3,Toronto CMA,3.0,8.7,2046.0,3.4,0.5,0
4,Ottawa-Gatineau CMA (Ont. part),3.0,15.4,1926.0,3.4,0.4,0
5,Calgary CMA,5.0,24.9,1914.0,NaN,0.2,0
6,Nanaimo CMA,2.2,23.6,1910.0,3.5,-0.7,0
7,Kitchener-Cambridge-Waterloo CMA,4.1,16.1,1832.0,3.3,0.5,0
8,Halifax CMA,2.7,9.0,1826.0,6.7,0.6,0
9,Guelph CMA,3.1,15.1,1802.0,4.6,1.2,0


## Query 2: Data-Driven Thresholds

**Business Question:** What are the thresholds for high rent (75th percentile), low vacancy (25th percentile), rapid rent growth (75th percentile), and vacancy tightening (25th percentile) for major Canadian rental markets in 2025?

In [3]:
q2_sql = """
WITH 
rent_ordered AS (
    SELECT average_rent_2br_2025 as val,
           ROW_NUMBER() OVER (ORDER BY average_rent_2br_2025) as rn,
           COUNT(*) OVER () as total
    FROM rental_market_2024_2025
    WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1 AND is_caution_quality = 0 AND average_rent_2br_2025 IS NOT NULL
),
rent_thresh AS (
    SELECT val as high_rent_threshold
    FROM rent_ordered
    WHERE rn = CAST(ROUND(1 + 0.75 * (total - 1)) AS INT)
),
vacancy_ordered AS (
    SELECT vacancy_rate_2025 as val,
           ROW_NUMBER() OVER (ORDER BY vacancy_rate_2025) as rn,
           COUNT(*) OVER () as total
    FROM rental_market_2024_2025
    WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1 AND is_caution_quality = 0 AND vacancy_rate_2025 IS NOT NULL
),
vacancy_thresh AS (
    SELECT val as low_vacancy_threshold
    FROM vacancy_ordered
    WHERE rn = CAST(ROUND(1 + 0.25 * (total - 1)) AS INT)
),
growth_ordered AS (
    SELECT rent_growth_pct_2024_2025 as val,
           ROW_NUMBER() OVER (ORDER BY rent_growth_pct_2024_2025) as rn,
           COUNT(*) OVER () as total
    FROM rental_market_2024_2025
    WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1 AND is_caution_quality = 0 AND rent_growth_pct_2024_2025 IS NOT NULL
),
growth_thresh AS (
    SELECT val as rapid_rent_growth_threshold
    FROM growth_ordered
    WHERE rn = CAST(ROUND(1 + 0.75 * (total - 1)) AS INT)
),
tightening_ordered AS (
    SELECT vacancy_change_pct_points as val,
           ROW_NUMBER() OVER (ORDER BY vacancy_change_pct_points) as rn,
           COUNT(*) OVER () as total
    FROM rental_market_2024_2025
    WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1 AND is_caution_quality = 0 AND vacancy_change_pct_points IS NOT NULL
),
tightening_thresh AS (
    SELECT val as vacancy_tightening_threshold
    FROM tightening_ordered
    WHERE rn = CAST(ROUND(1 + 0.25 * (total - 1)) AS INT)
)
SELECT 
    (SELECT high_rent_threshold FROM rent_thresh) as high_rent_threshold,
    (SELECT low_vacancy_threshold FROM vacancy_thresh) as low_vacancy_threshold,
    (SELECT rapid_rent_growth_threshold FROM growth_thresh) as rapid_rent_growth_threshold,
    (SELECT vacancy_tightening_threshold FROM tightening_thresh) as vacancy_tightening_threshold;
"""

df_thresholds = pd.read_sql_query(q2_sql, conn)
df_thresholds.to_csv('../outputs/tables/rental_market_thresholds.csv', index=False)
print("Calculated Thresholds:")
df_thresholds

Calculated Thresholds:


,high_rent_threshold,low_vacancy_threshold,rapid_rent_growth_threshold,vacancy_tightening_threshold
0,1802.0,2.5,6.1,0.1


## Query 3: Rental Market Pressure Segmentation

**Business Question:** Which markets show the strongest signs of rental-market pressure in 2025 based on thresholds for high rent, low vacancy, rapid rent growth, and vacancy tightening?

In [4]:
q3_sql = """
WITH 
thresholds AS (
    WITH 
    rent_ordered AS (
        SELECT average_rent_2br_2025 as val,
               ROW_NUMBER() OVER (ORDER BY average_rent_2br_2025) as rn,
               COUNT(*) OVER () as total
        FROM rental_market_2024_2025
        WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1 AND is_caution_quality = 0 AND average_rent_2br_2025 IS NOT NULL
    ),
    rent_thresh AS (
        SELECT val as high_rent_threshold
        FROM rent_ordered
        WHERE rn = CAST(ROUND(1 + 0.75 * (total - 1)) AS INT)
    ),
    vacancy_ordered AS (
        SELECT vacancy_rate_2025 as val,
               ROW_NUMBER() OVER (ORDER BY vacancy_rate_2025) as rn,
               COUNT(*) OVER () as total
        FROM rental_market_2024_2025
        WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1 AND is_caution_quality = 0 AND vacancy_rate_2025 IS NOT NULL
    ),
    vacancy_thresh AS (
        SELECT val as low_vacancy_threshold
        FROM vacancy_ordered
        WHERE rn = CAST(ROUND(1 + 0.25 * (total - 1)) AS INT)
    ),
    growth_ordered AS (
        SELECT rent_growth_pct_2024_2025 as val,
               ROW_NUMBER() OVER (ORDER BY rent_growth_pct_2024_2025) as rn,
               COUNT(*) OVER () as total
        FROM rental_market_2024_2025
        WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1 AND is_caution_quality = 0 AND rent_growth_pct_2024_2025 IS NOT NULL
    ),
    growth_thresh AS (
        SELECT val as rapid_rent_growth_threshold
        FROM growth_ordered
        WHERE rn = CAST(ROUND(1 + 0.75 * (total - 1)) AS INT)
    ),
    tightening_ordered AS (
        SELECT vacancy_change_pct_points as val,
               ROW_NUMBER() OVER (ORDER BY vacancy_change_pct_points) as rn,
               COUNT(*) OVER () as total
        FROM rental_market_2024_2025
        WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1 AND is_caution_quality = 0 AND vacancy_change_pct_points IS NOT NULL
    ),
    tightening_thresh AS (
        SELECT val as vacancy_tightening_threshold
        FROM tightening_ordered
        WHERE rn = CAST(ROUND(1 + 0.25 * (total - 1)) AS INT)
    )
    SELECT 
        (SELECT high_rent_threshold FROM rent_thresh) as high_rent_threshold,
        (SELECT low_vacancy_threshold FROM vacancy_thresh) as low_vacancy_threshold,
        (SELECT rapid_rent_growth_threshold FROM growth_thresh) as rapid_rent_growth_threshold,
        (SELECT vacancy_tightening_threshold FROM tightening_thresh) as vacancy_tightening_threshold
),
scored_centres AS (
    SELECT 
        m.centre,
        m.vacancy_rate_2024,
        m.vacancy_rate_2025,
        m.vacancy_change_pct_points,
        m.turnover_rate_2024,
        m.turnover_rate_2025,
        m.average_rent_2br_2024,
        m.average_rent_2br_2025,
        m.rent_growth_pct_2024_2025,
        m.is_caution_quality,
        (CASE WHEN m.average_rent_2br_2025 >= t.high_rent_threshold THEN 1 ELSE 0 END) +
        (CASE WHEN m.vacancy_rate_2025 <= t.low_vacancy_threshold THEN 1 ELSE 0 END) +
        (CASE WHEN m.rent_growth_pct_2024_2025 >= t.rapid_rent_growth_threshold THEN 1 ELSE 0 END) +
        (CASE WHEN m.vacancy_change_pct_points <= t.vacancy_tightening_threshold AND m.vacancy_change_pct_points < 0 THEN 1 ELSE 0 END) as market_pressure_score,
        (m.average_rent_2br_2025 >= t.high_rent_threshold) as is_high_rent,
        (m.vacancy_rate_2025 <= t.low_vacancy_threshold) as is_low_vacancy,
        (m.rent_growth_pct_2024_2025 >= t.rapid_rent_growth_threshold) as is_rapid_growth,
        (m.vacancy_change_pct_points <= t.vacancy_tightening_threshold AND m.vacancy_change_pct_points < 0) as is_vacancy_tightening
    FROM rental_market_2024_2025 m
    CROSS JOIN thresholds t
    WHERE m.has_valid_rent_2025 = 1 AND m.has_valid_vacancy_2025 = 1
)
SELECT 
    centre,
    vacancy_rate_2024,
    vacancy_rate_2025,
    vacancy_change_pct_points,
    turnover_rate_2024,
    turnover_rate_2025,
    average_rent_2br_2024,
    average_rent_2br_2025,
    rent_growth_pct_2024_2025,
    market_pressure_score,
    CASE 
        WHEN is_caution_quality = 1 THEN 'Caution-quality data'
        WHEN is_high_rent = 1 AND is_low_vacancy = 1 THEN 'High rent + low vacancy'
        WHEN is_low_vacancy = 1 AND is_rapid_growth = 1 THEN 'Low vacancy + rapid rent growth'
        WHEN is_high_rent = 1 AND is_rapid_growth = 1 THEN 'High rent + rapid rent growth'
        WHEN is_low_vacancy = 1 AND is_vacancy_tightening = 1 THEN 'Low vacancy + vacancy tightening'
        WHEN is_high_rent = 1 THEN 'High rent'
        WHEN is_rapid_growth = 1 THEN 'Rapid rent growth'
        WHEN is_vacancy_tightening = 1 THEN 'Vacancy tightening'
        ELSE 'Monitor'
    END as market_pressure_label,
    CASE 
        WHEN market_pressure_score = 0 THEN 'Stable or moderate conditions. Monitor.'
        ELSE rtrim(
            (CASE WHEN is_high_rent = 1 THEN 'High 2025 rent; ' ELSE '' END) ||
            (CASE WHEN is_low_vacancy = 1 THEN 'Low 2025 vacancy; ' ELSE '' END) ||
            (CASE WHEN is_rapid_growth = 1 THEN 'Rapid rent growth; ' ELSE '' END) ||
            (CASE WHEN is_vacancy_tightening = 1 THEN 'Vacancy tightening; ' ELSE '' END),
            '; '
        )
    END as pressure_reason,
    is_caution_quality
FROM scored_centres
ORDER BY market_pressure_score DESC, average_rent_2br_2025 DESC, vacancy_rate_2025 ASC;
"""

df_segments = pd.read_sql_query(q3_sql, conn)
df_segments.to_csv('../outputs/tables/rental_market_pressure_segments.csv', index=False)
print(f"Exported {len(df_segments)} rows to outputs/tables/rental_market_pressure_segments.csv")
df_segments.head(15)

Exported 43 rows to outputs/tables/rental_market_pressure_segments.csv


,centre,vacancy_rate_2024,vacancy_rate_2025,vacancy_change_pct_points,turnover_rate_2024,turnover_rate_2025,average_rent_2br_2024,average_rent_2br_2025,rent_growth_pct_2024_2025,market_pressure_score,market_pressure_label,pressure_reason,is_caution_quality
0,Nanaimo CMA,2.9,2.2,-0.7,20.2,23.6,1787.0,1910.0,3.5,3,High rent + low vacancy,High 2025 rent; Low 2025 vacancy; Vacancy tigh...,0
1,St. John's CMA,2.1,2.0,-0.1,14.1,13.0,1250.0,1361.0,7.6,3,Low vacancy + rapid rent growth,Low 2025 vacancy; Rapid rent growth; Vacancy t...,0
2,Saguenay CMA,1.6,1.3,-0.3,13.3,12.6,882.0,985.0,10.9,3,Low vacancy + rapid rent growth,Low 2025 vacancy; Rapid rent growth; Vacancy t...,0
3,Halifax CMA,2.1,2.7,0.6,10.0,9.0,1707.0,1826.0,6.7,2,High rent + rapid rent growth,High 2025 rent; Rapid rent growth,0
4,Kingston CMA,2.9,2.4,-0.5,16.4,17.3,1676.0,1764.0,2.4,2,Low vacancy + vacancy tightening,Low 2025 vacancy; Vacancy tightening,0
5,Kamloops CMA,1.4,1.2,-0.2,14.7,17.6,1531.0,1679.0,NaN,2,Low vacancy + vacancy tightening,Low 2025 vacancy; Vacancy tightening,0
6,Greater Sudbury/Grand Sudbury CMA,1.5,1.2,-0.3,8.3,6.6,1462.0,1558.0,NaN,2,Low vacancy + vacancy tightening,Low 2025 vacancy; Vacancy tightening,0
7,Saint John CMA,4.0,1.9,-2.1,13.8,11.8,1229.0,1290.0,5.8,2,Low vacancy + vacancy tightening,Low 2025 vacancy; Vacancy tightening,0
8,Québec CMA,0.9,2.4,1.5,13.9,13.0,1159.0,1277.0,6.1,2,Low vacancy + rapid rent growth,Low 2025 vacancy; Rapid rent growth,0
9,Vancouver CMA,1.6,3.7,2.1,9.1,11.6,2314.0,2363.0,2.2,1,High rent,High 2025 rent,0


## Query 4: 2024–2025 Change Summary

**Business Question:** How did rent and vacancy conditions change between 2024 and 2025 across centres?

In [5]:
q4_sql = """
SELECT 
    centre,
    average_rent_2br_2024,
    average_rent_2br_2025,
    rent_growth_pct_2024_2025,
    vacancy_rate_2024,
    vacancy_rate_2025,
    vacancy_change_pct_points,
    turnover_rate_2024,
    turnover_rate_2025,
    CASE 
        WHEN rent_growth_pct_2024_2025 > 0 AND vacancy_change_pct_points < 0 THEN 'Rent rising + vacancy tightening'
        WHEN rent_growth_pct_2024_2025 > 0 AND vacancy_change_pct_points > 0 THEN 'Rent rising + vacancy easing'
        WHEN (rent_growth_pct_2024_2025 <= 0 OR rent_growth_pct_2024_2025 IS NULL) AND vacancy_change_pct_points < 0 THEN 'Rent stable/unknown + vacancy tightening'
        ELSE 'Monitor'
    END as change_label,
    is_caution_quality
FROM rental_market_2024_2025
WHERE has_valid_rent_2025 = 1 AND has_valid_vacancy_2025 = 1;
"""

df_change = pd.read_sql_query(q4_sql, conn)
df_change.to_csv('../outputs/tables/rental_market_change_summary.csv', index=False)
print(f"Exported {len(df_change)} rows to outputs/tables/rental_market_change_summary.csv")
df_change.head(15)

Exported 43 rows to outputs/tables/rental_market_change_summary.csv


,centre,average_rent_2br_2024,average_rent_2br_2025,rent_growth_pct_2024_2025,vacancy_rate_2024,vacancy_rate_2025,vacancy_change_pct_points,turnover_rate_2024,turnover_rate_2025,change_label,is_caution_quality
0,St. John's CMA,1250.0,1361.0,7.6,2.1,2.0,-0.1,14.1,13.0,Rent rising + vacancy tightening,0
1,Charlottetown CA,1196.0,1300.0,4.7,0.7,1.6,0.9,9.1,9.8,Rent rising + vacancy easing,1
2,Halifax CMA,1707.0,1826.0,6.7,2.1,2.7,0.6,10.0,9.0,Rent rising + vacancy easing,0
3,Fredericton CMA,1341.0,1413.0,5.2,0.9,2.5,1.6,18.1,13.1,Rent rising + vacancy easing,0
4,Moncton CMA,1353.0,1453.0,4.7,1.5,3.9,2.4,13.4,15.4,Rent rising + vacancy easing,0
5,Saint John CMA,1229.0,1290.0,5.8,4.0,1.9,-2.1,13.8,11.8,Rent rising + vacancy tightening,0
6,Saguenay CMA,882.0,985.0,10.9,1.6,1.3,-0.3,13.3,12.6,Rent rising + vacancy tightening,0
7,Drummondville CMA,888.0,1045.0,NaN,1.5,1.8,0.3,14.3,10.9,Monitor,0
8,Montréal CMA,1176.0,1346.0,7.2,2.1,2.9,0.8,11.1,9.4,Rent rising + vacancy easing,0
9,Ottawa-Gatineau CMA (Qué. part),1353.0,1460.0,4.7,1.9,3.8,1.9,17.3,16.2,Rent rising + vacancy easing,1


## Query 5: Quality Review

**Business Question:** Which centres have data quality concerns (caution-quality estimate flags or missing rent/vacancy/growth measures)?

In [6]:
q5_sql = """
SELECT 
    centre,
    CASE 
        WHEN has_valid_rent_2025 = 0 THEN 'Missing 2025 average rent'
        WHEN has_valid_vacancy_2025 = 0 THEN 'Missing 2025 vacancy rate'
        WHEN is_caution_quality = 1 AND has_valid_rent_growth = 0 THEN 'Caution quality (flag d) & Missing rent growth'
        WHEN is_caution_quality = 1 THEN 'Caution quality (flag d)'
        WHEN has_valid_rent_growth = 0 THEN 'Missing rent growth'
        ELSE 'Other quality concern'
    END as quality_review_reason,
    is_caution_quality,
    has_valid_rent_2025,
    has_valid_vacancy_2025,
    has_valid_rent_growth,
    vacancy_quality_2025,
    rent_quality_2025,
    rent_growth_quality_2025
FROM rental_market_2024_2025
WHERE is_caution_quality = 1 
   OR has_valid_rent_growth = 0 
   OR has_valid_vacancy_2025 = 0 
   OR has_valid_rent_2025 = 0;
"""

df_quality = pd.read_sql_query(q5_sql, conn)
df_quality.to_csv('../outputs/tables/rental_market_quality_review.csv', index=False)
print(f"Exported {len(df_quality)} rows to outputs/tables/rental_market_quality_review.csv")
df_quality.head(15)

Exported 14 rows to outputs/tables/rental_market_quality_review.csv


,centre,quality_review_reason,is_caution_quality,has_valid_rent_2025,has_valid_vacancy_2025,has_valid_rent_growth,vacancy_quality_2025,rent_quality_2025,rent_growth_quality_2025
0,Charlottetown CA,Caution quality (flag d),1,1,1,1,c,a,d
1,Drummondville CMA,Missing rent growth,0,1,1,0,c,b,None
2,Ottawa-Gatineau CMA (Qué. part),Caution quality (flag d),1,1,1,1,c,a,d
3,Belleville - Quinte West CMA,Missing rent growth,0,1,1,0,c,a,None
4,Brantford CMA,Missing rent growth,0,1,1,0,c,a,None
5,Oshawa CMA,Missing rent growth,0,1,1,0,b,a,None
6,Peterborough CMA,Missing rent growth,0,1,1,0,b,a,None
7,Greater Sudbury/Grand Sudbury CMA,Missing rent growth,0,1,1,0,a,a,None
8,Windsor CMA,Caution quality (flag d),1,1,1,1,b,a,d
9,Calgary CMA,Missing rent growth,0,1,1,0,a,a,None


## Summary of Pipeline Execution

In [7]:
# 1. Total centres analyzed
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM rental_market_2024_2025;")
total_centres = cursor.fetchone()[0]

# 2. Valid rent & vacancy records
valid_records = len(df_rankings)

# 3. Caution quality records
cursor.execute("SELECT COUNT(*) FROM rental_market_2024_2025 WHERE is_caution_quality = 1;")
caution_records = cursor.fetchone()[0]

print("=================== PIPELINE EXECUTION SUMMARY ===================")
print(f"Total centres in database:           {total_centres}")
print(f"Valid 2025 rent & vacancy records:   {valid_records}")
print(f"Caution quality records:             {caution_records}")

# 4. Thresholds used
print("\n--- Thresholds (Calculated on non-caution valid records only) ---")
for col in df_thresholds.columns:
    print(f"{col:<32}: {df_thresholds.iloc[0][col]}")

# 5. Distribution of pressure labels
print("\n--- Distribution of Market Pressure Labels ---")
print(df_segments['market_pressure_label'].value_counts())

# 6. Top 10 non-caution centres by market_pressure_score, then highest rent, then lowest vacancy
print("\n--- Top 10 Non-Caution Centres by Market Pressure ---")
df_top_10 = df_segments[df_segments['is_caution_quality'] == 0].head(10)
print(df_top_10[['centre', 'market_pressure_score', 'average_rent_2br_2025', 'vacancy_rate_2025', 'market_pressure_label']])

print("==================================================================")

conn.close()

=================== PIPELINE EXECUTION SUMMARY ===================
Total centres in database:           43
Valid 2025 rent & vacancy records:   43
Caution quality records:             5

--- Thresholds (Calculated on non-caution valid records only) ---
high_rent_threshold             : 1802.0
low_vacancy_threshold           : 2.5
rapid_rent_growth_threshold     : 6.1
vacancy_tightening_threshold    : 0.1

--- Distribution of Market Pressure Labels ---
market_pressure_label
Monitor                             15
High rent                            8
Caution-quality data                 5
Low vacancy + vacancy tightening     4
Rapid rent growth                    4
Low vacancy + rapid rent growth      3
Vacancy tightening                   2
High rent + low vacancy              1
High rent + rapid rent growth        1
Name: count, dtype: int64

--- Top 10 Non-Caution Centres by Market Pressure ---
                              centre  market_pressure_score  \
0                        Na